In [2]:
import numpy as np

def fill_label_ints(present_keys_ints: list[int], fill_upto: int):
    present_keys_ints = np.sort(present_keys_ints)
    if fill_upto < len(present_keys_ints):
        raise ValueError("fill_upto must be greater than or equal to the number of present basis states.")
    elif fill_upto >= present_keys_ints[-1]:
        return np.arange(fill_upto)
    else:
        to_add = fill_upto - len(present_keys_ints)
        ints_between_keys = np.diff(present_keys_ints) - 1
        ints_between_keys_cumsum = np.cumsum(ints_between_keys)
        if ints_between_keys_cumsum[-1] > to_add:
            max_key_to_fill_between_index = np.argwhere(ints_between_keys_cumsum >= to_add)[0][0] + 1
            max_key_to_fill_between = present_keys_ints[max_key_to_fill_between_index]
            all_to_fill_between = np.arange(present_keys_ints[0]+1, max_key_to_fill_between)
            present_elements = present_keys_ints[:max_key_to_fill_between_index]
            right_elements = present_keys_ints[max_key_to_fill_between_index:]
        else:
            all_to_fill_between = np.arange(present_keys_ints[0]+1, present_keys_ints[-1])
            present_elements = present_keys_ints
            right_elements = np.array([], dtype=int)
        to_fill_between = np.setdiff1d(all_to_fill_between, present_elements)[:to_add]
        between_elements = np.union1d(present_elements, to_fill_between)
        nfilled_between = len(to_fill_between)
        if nfilled_between < to_add:
            to_fill_before = np.arange(to_add - nfilled_between)
        else:
            to_fill_before = np.array([], dtype=int)
        return np.concatenate((to_fill_before, between_elements, right_elements))        

In [3]:
a = [100, 102, 120]

num_keep = 8

fill_label_ints(a, num_keep)

array([100, 101, 102, 103, 104, 105, 106, 120])

In [5]:
import utils.dec as dec

nqubits = 8

dummydict = {f"{i:0{nqubits}b}":0 for i in a}
filled_labels = dec.fill_counts_range(dummydict, dummydict, num_keep, nqubits)[0]
[int(label, 2) for label in filled_labels]

[100, 101, 102, 103, 104, 105, 106, 120]

## Test

In [6]:
def generate_random_vectors(max_len, max_element, num_vectors):
    vectors = []
    for i in range(num_vectors):
        length = np.random.randint(1, max_len + 1)
        vector = np.random.randint(0, max_element, length)
        vectors.append(vector)
    return vectors

def opt_test(max_len, max_element, num_vectors):
    vectors = generate_random_vectors(max_len, max_element, num_vectors)
    passed_tests = np.zeros(num_vectors, dtype=bool)
    for i, vector in enumerate(vectors):
        # Traditional approach
        nqubits = int(np.ceil(np.log2(max_element)))
        dummydict = {f"{i:0{nqubits}b}":0 for i in vector}
        filled_labels = dec.fill_counts_range(dummydict, dummydict, 2**nqubits, nqubits)[0]
        filled_labels_ints_traditional = np.sort([int(label, 2) for label in filled_labels])
        # Optimized approach
        filled_labels_ints_optimized = np.sort(fill_label_ints(vector, 2**nqubits))
        # Compare results
        passed_tests[i] = np.array_equal(filled_labels_ints_traditional, filled_labels_ints_optimized)
    return passed_tests

In [7]:
np.all(opt_test(2**14, 2**14, 1000))

True

In [8]:
import scipy.sparse as sparse

nelements = 100
max_int = 1000
shape = 2**150

random_data_a = np.random.randint(0, max_int, nelements)
random_data_b = np.random.randint(0, max_int, nelements)

random_data_positions_a = np.concatenate(([0], np.random.randint(0, nelements, size=(nelements-1,))))
random_data_positions_b = np.concatenate(([0], np.random.randint(0, nelements, size=(nelements-1,))))

a = sparse.dok_array((shape,), dtype=int)

In [9]:
a

<1427247692705959881058285969449495136382746624 sparse array of type '<class 'numpy.int64'>'
	with 0 stored elements in Dictionary Of Keys format>

In [10]:
import scipy.sparse as sparse

nelements = 100
max_int = 1000
shape = 2**127

random_data_a = np.random.randint(0, max_int, nelements)
random_data_b = np.random.randint(0, max_int, nelements)

random_data_positions_a = np.concatenate(([0], np.random.randint(0, nelements, size=(nelements-1,))))
random_data_positions_b = np.concatenate(([0], np.random.randint(0, nelements, size=(nelements-1,))))

a = sparse.dok_array((shape,), dtype=int)
b = sparse.dok_array((shape,), dtype=int)

for i, pos in enumerate(random_data_positions_a):
    a[pos] = random_data_a[i]

for i, pos in enumerate(random_data_positions_b):
    b[pos] = random_data_b[i]

In [11]:
print(a[0])
print(b[0])
print((a+b)[0])

116
696
812


## What happens for larger sizes

In [12]:
curated_data_filepath_small = '../notebooks/Z2/Curated Data/z2chainquench_curated_data_ibm_marrakesh_L_13_J_2_h_0.8_lamb_1_g_zero_mean_final_time_5_dt_0.125_steps_40_shots_75000_ppos_3_pl_6_ft_None_DD_True_PT_True_dec.pkl'
curated_data_filepath_large = '../notebooks/Z2/Curated Data/z2chainquench_curated_data_ibm_marrakesh_L_21_J_2_h_0.8_lamb_1_g_zero_mean_final_time_5_dt_0.125_steps_40_shots_75000_ppos_7_pl_6_ft_None_DD_True_PT_True_dec.pkl'

In [13]:
import pickle

L_small = 13
shots_small = 75000
with open(curated_data_filepath_small, "rb") as f:
    observables_small, samples_dicts_small, postselected_samples_dicts_small, layout = pickle.load(f)

L_large = 21
shots_large = 75000
with open(curated_data_filepath_large, "rb") as f:
    observables_large, samples_dicts_large, postselected_samples_dicts_large, layout = pickle.load(f)

In [22]:
import utils.dec_opt as decopt

job_index = 0

result_small = decopt.mitigate(samples_dicts_small[1][job_index], samples_dicts_small[2][job_index], {"0"*(2*L_small-1):shots_small})

In [17]:
len(result_small)

541

In [18]:
len(samples_dicts_small[1][job_index])

53443